# ODIR-5K Eye Disease Dataset — Exploratory Data Analysis
**Project:** Multi-Class Eye Disease Prediction Using EfficientNetB4  
**Author:** Aiyesha Rukhsar | M.Tech, NIT Delhi  
**Dataset:** ODIR-5K (Ocular Disease Intelligent Recognition)

In [ ]:
import os, sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from PIL import Image

import config
from src.preprocess import load_odir_annotations, build_image_df, LABEL_COLUMNS
from src.utils import plot_label_distribution, visualize_sample_images

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline
print('All imports OK.')

## 1. Load annotations

In [ ]:
df = load_odir_annotations(config.ODIR_ANNOTATIONS)
print(f'Shape: {df.shape}')
df.head()

## 2. Label distribution

In [ ]:
print('Label counts:')
print(df[LABEL_COLUMNS].sum())

print('\nMulti-label statistics:')
df['num_labels'] = df[LABEL_COLUMNS].sum(axis=1)
print(df['num_labels'].value_counts().sort_index())

In [ ]:
plot_label_distribution(df, LABEL_COLUMNS, save_path='../results/label_distribution.png')

## 3. Build image DataFrame and check file counts

In [ ]:
image_df = build_image_df(df, config.ODIR_TRAIN_IMAGES)
print(f'Total images: {len(image_df)}')
print(f'Left eye : {len(image_df[image_df.eye == "left"])}')
print(f'Right eye: {len(image_df[image_df.eye == "right"])}')
image_df.head()

## 4. Sample images per class

In [ ]:
visualize_sample_images(image_df, save_path='../results/sample_images.png')

## 5. Image size distribution

In [ ]:
import random
sample_paths = random.sample(image_df['image_path'].tolist(), min(200, len(image_df)))

heights, widths = [], []
for p in sample_paths:
    img = cv2.imread(p)
    if img is not None:
        heights.append(img.shape[0])
        widths.append(img.shape[1])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(heights, bins=20, color='steelblue', edgecolor='k')
ax1.set_title('Image Height Distribution'); ax1.set_xlabel('Pixels')
ax2.hist(widths,  bins=20, color='salmon',   edgecolor='k')
ax2.set_title('Image Width Distribution');  ax2.set_xlabel('Pixels')
plt.suptitle('ODIR-5K Fundus Image Dimensions', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('../results/image_size_dist.png', dpi=120)
plt.show()

print(f'Height — mean:{np.mean(heights):.0f}  std:{np.std(heights):.0f}')
print(f'Width  — mean:{np.mean(widths):.0f}   std:{np.std(widths):.0f}')

## 6. Label co-occurrence heatmap

In [ ]:
cooccur = df[LABEL_COLUMNS].T.dot(df[LABEL_COLUMNS])

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cooccur, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=config.DISEASE_LABELS,
            yticklabels=config.DISEASE_LABELS, ax=ax)
ax.set_title('Label Co-occurrence Matrix — ODIR-5K', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('../results/cooccurrence_matrix.png', dpi=120)
plt.show()

## 7. Ben Graham preprocessing visualisation

In [ ]:
from src.preprocess import load_and_preprocess_image

sample_path = image_df['image_path'].iloc[0]

raw = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)
raw = cv2.resize(raw, (380, 380))
enhanced = load_and_preprocess_image(sample_path, apply_ben_graham=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.imshow(raw);      ax1.set_title('Original',              fontweight='bold'); ax1.axis('off')
ax2.imshow(enhanced); ax2.set_title('Ben Graham Preprocessed', fontweight='bold'); ax2.axis('off')
plt.suptitle('Preprocessing Comparison', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('../results/preprocessing_comparison.png', dpi=120)
plt.show()

## 8. Summary statistics

In [ ]:
print('=' * 50)
print('ODIR-5K DATASET SUMMARY')
print('=' * 50)
print(f'Total patients   : {len(df)}')
print(f'Total images     : {len(image_df)}')
print(f'Image size target: {config.IMAGE_SIZE}')
print(f'Number of classes: {config.NUM_CLASSES}')
print()
for code, label in zip(config.DISEASE_CODES, config.DISEASE_LABELS):
    n = int(df[code].sum())
    pct = n / len(df) * 100
    print(f'  [{code}] {label:<30} {n:>5} ({pct:5.1f}%)')